# LoRA safety reviewer: baseline → training → evaluation

Steps 2–4 of `experiments/lora-reviewer/README.md`, in Google Colab.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom
(Shift+Enter). Each cell says roughly how long it takes.

What happens:
1. setup: libraries and the public repository (for the production reviewer prompt and the eval cases);
2. the dataset: upload `train.jsonl` and `val.jsonl`;
3. the base model (Qwen2.5-7B-Instruct, 4-bit) is measured **before** training;
4. LoRA training (QLoRA);
5. the same measurement **after** training, next to the production reviewer (Haiku).

Two measurements, the same as `evals/safety/run_safety_evals.py`:
- **safety evals**: 15 unsafe and 12 safe hand-labelled activities. Recall on unsafe must be 100%:
  a reviewer that misses a hazard is not a candidate, however cheap;
- **val**: agreement with the teacher (Sonnet) on rows the student never saw. The teacher agreed with
  itself on 87% of doubled rows, so that is the ceiling.

## 1. Setup (~3 min)

In [ ]:
!pip install -q transformers==4.53.2 peft==0.16.0 trl==0.19.1 bitsandbytes==0.46.1 accelerate==1.8.1 datasets==3.6.0
!git clone -q https://github.com/LukshinVI/game-of-the-day.git repo 2>/dev/null || git -C repo pull -q
import sys
sys.path.insert(0, "repo")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 2. Dataset
Upload the two files sent in Telegram: `train.jsonl` and `val.jsonl`.

In [ ]:
import json, os
from google.colab import files
if not (os.path.exists("train.jsonl") and os.path.exists("val.jsonl")):
    files.upload()
train_rows = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
val_rows = [json.loads(l) for l in open("val.jsonl", encoding="utf-8")]
print("train:", len(train_rows), " val:", len(val_rows))
print(train_rows[0]["completion"][0]["content"][:300])

## 3. Base model and the measurement (~15–20 min)
`MODEL` can be switched to `Qwen/Qwen2.5-3B-Instruct`: twice as fast, likely weaker.

In [ ]:
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.preparation import review

MODEL = "Qwen/Qwen2.5-7B-Instruct"
FORMAT = "\n" + train_rows[0]["prompt"][0]["content"].rsplit("\n", 1)[1]   # the same answer-format line as in training

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))
model.config.use_cache = True


def ask(prompts, batch=4):
    """Greedy answers for a list of user prompts."""
    out = []
    for i in range(0, len(prompts), batch):
        texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                               add_generation_prompt=True) for p in prompts[i:i + batch]]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        out += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  {len(out)}/{len(prompts)}", end="\r")
    return out


def parse(text):
    """(blocks, fixes) from an answer; an unparseable answer is (None, None)."""
    m = re.search(r"\{.*\}", text, re.S)
    try:
        data = json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        data = None
    if not isinstance(data, dict):
        return None, None
    clean = lambda k: [str(x).strip() for x in (data.get(k) or []) if str(x).strip()][:3]
    return clean("block"), clean("fix")


cases = json.load(open("repo/evals/safety/cases.json", encoding="utf-8"))["cases"]


def case_prompt(case):
    activity = {k: case["activity"].get(k) for k in ("title", "materials", "steps", "safety", "duration_minutes")}
    return review.TIERED_PROMPT.format(age=case["age_months"],
                                       activity=json.dumps(activity, ensure_ascii=False, indent=2)) + FORMAT


def measure(name):
    """The safety evals (model alone and with the production rules) and agreement on val."""
    answers = [parse(a) for a in ask([case_prompt(c) for c in cases])]
    unsafe = [(c, a) for c, a in zip(cases, answers) if c["label"] == "unsafe"]
    safe = [(c, a) for c, a in zip(cases, answers) if c["label"] == "safe"]
    broken = sum(1 for a in answers if a[0] is None)
    blocked = lambda a, rules: a[0] is not None and bool(review.split(a[0], a[1])[0] if rules else a[0])
    missed = [c["id"] for c, a in unsafe if not blocked(a, True)]
    alarms = [c["id"] for c, a in safe if blocked(a, True)]

    val_answers = [parse(a) for a in ask([r["prompt"][0]["content"] for r in val_rows])]
    verdict = lambda a: None if a[0] is None else ("block" if a[0] else "not block")
    teacher = lambda r: "block" if r["verdict"] == "block" else "not block"
    agree = sum(verdict(a) == teacher(r) for a, r in zip(val_answers, val_rows))

    print(f"\n=== {name} ===")
    print(f"unsafe caught, model alone: {sum(blocked(a, False) for _, a in unsafe)}/{len(unsafe)}; "
          f"with the production rules: {len(unsafe) - len(missed)}/{len(unsafe)}")
    print(f"safe blocked, model alone: {sum(blocked(a, False) for _, a in safe)}/{len(safe)}; "
          f"with the rules: {len(alarms)}/{len(safe)}")
    print(f"answers that were not valid JSON: {broken}/{len(cases)}")
    print(f"val: agrees with the teacher (block / not block) on {agree}/{len(val_rows)} = {agree / len(val_rows):.0%}")
    print("missed:", missed or "none", "| false alarms:", alarms or "none")
    print("Production reviewer (Haiku, 18.11, 3 runs): unsafe 98%, safe blocked 11% (all by the rules, 0 by the model).")
    return {"name": name, "missed": missed, "alarms": alarms, "broken": broken, "val_agree": agree,
            "answers": answers}


before = measure("base model, before training")

## 4. Training (~40–60 min on T4)
QLoRA: the base model stays frozen in 4-bit, only small adapter matrices are trained.
- `r` — the adapter rank (its capacity); `lora_alpha` — its scale (usually 2×r);
- the loss is computed on the answer only (prompt-completion format);
- 2 epochs: with ~350 examples more would start memorising.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model.config.use_cache = False
train_ds = Dataset.from_list([{"prompt": r["prompt"], "completion": r["completion"]} for r in train_rows])

config = SFTConfig(
    output_dir="out", num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    per_device_train_batch_size=1, gradient_accumulation_steps=8, gradient_checkpointing=True,
    fp16=True, max_length=2048, logging_steps=5, save_strategy="no", report_to="none",
)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
trainer = SFTTrainer(model=model, args=config, train_dataset=train_ds, processing_class=tokenizer, peft_config=lora)
trainer.model.print_trainable_parameters()
trainer.train()
model = trainer.model
model.config.use_cache = True

## 5. After training (~15–20 min)

In [ ]:
after = measure("with the LoRA adapter")
print()
for r in (before, after):
    print(f"{r['name']:30} missed {len(r['missed'])}, false alarms {len(r['alarms'])}, "
          f"invalid JSON {r['broken']}, val agreement {r['val_agree']}/{len(val_rows)}")

## 6. Save the adapter
About 100-150 MB: the adapter only, not the 15 GB model.

In [ ]:
trainer.save_model("adapter")
json.dump({"before": {k: v for k, v in before.items() if k != "answers"},
           "after": {k: v for k, v in after.items() if k != "answers"}},
          open("adapter/results.json", "w"), ensure_ascii=False, indent=2)
!zip -qr adapter.zip adapter
files.download("adapter.zip")